In [ ]:
# !pip install sentence-transformers

  Using cached sentence_transformers-5.3.0-py3-none-any.whl.metadata (16 kB)
Using cached sentence_transformers-5.3.0-py3-none-any.whl (512 kB)
   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
   -------- ------------------------------- 2.1/10.2 MB 11.8 MB/s eta 0:00:01
   ------------------ --------------------- 4.7/10.2 MB 11.9 MB/s eta 0:00:01
   --------------------------- ------------ 7.1/10.2 MB 11.8 MB/s eta 0:00:01
   ------------------------------------- -- 9.7/10.2 MB 11.6 MB/s eta 0:00:01
   ---------------------------------------- 10.2/10.2 MB 11.2 MB/s  0:00:00

  Attempting uninstall: regex

    Found existing installation: regex 2024.11.6

    Uninstalling regex-2024.11.6:

      Successfully uninstalled regex-2024.11.6

   ---------------------------------------- 0/3 [regex]
   ------------- -------------------------- 1/3 [transformers]
   ------------- -------------------------- 1/3 [transformers]
   ------------- -------------------------- 1/3 [tr

In [16]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser 


In [10]:

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.load_local(
    'faiss_index',  # folder path jahan download kiya
    embeddings,
    allow_dangerous_deserialization=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
all_docs = list(db.docstore._dict.values())
retriever = db.as_retriever(search_kwargs={"k": 3,"fetch_k":25,"lambda_mult":0.75},search_type="mmr")


In [14]:
prompt = ChatPromptTemplate.from_template(
"""
You are a legal assistant designed to help non-lawyers understand legal documents.

Use ONLY the provided context to answer the question.

IMPORTANT RULES:
- Do NOT use prior knowledge outside the context
- If the answer is not clearly stated, say: "Not found in the document."
- Explain in simple, plain English (avoid legal jargon)
- Be accurate and do not guess

YOUR TASK:
1. Explain what the relevant part of the document means
2. Highlight any obligations, risks, or important conditions
3. If applicable, explain what the user is agreeing to

Context:
{context}

Question:
{question}

Answer (in simple terms):
"""
)

In [24]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma3:1b", 
    temperature=0.4
)


In [29]:
from langchain_ollama import ChatOllama

llm2 = ChatOllama(
    model="mistral:latest", 
    temperature=0.4
)

In [34]:
from langchain_ollama import ChatOllama

llm3 = ChatOllama(
    model="llama3.1:8b", 
    temperature=0.4
)

In [38]:

llm4 = ChatOllama(
    model="phi3:latest", 
    temperature=0.4
)

In [25]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt 
    | llm
    | StrOutputParser()
)

In [31]:
rag_chain2 = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt 
    | llm2
    | StrOutputParser()
)

In [35]:
rag_chain3 = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt 
    | llm3
    | StrOutputParser()
)

In [39]:
rag_chain4 = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt 
    | llm4
    | StrOutputParser()
)

In [ ]:
query1 = "What are Personal Liberty: Procedure Established by Law?"

response = rag_chain.invoke(query1)
print(response)

The text states: “The view expressed in A. K. Gopalan’s case was revisited in this case after about 28 years. The main issues were whether the right to go abroad is a part of the right to personal liberty under Article 21 and whether the Passport Act prescribes a ‘procedure’ as required by Article 21 of the Constitution.”

Therefore, Personal Liberty: Procedure Established by Law refers to the right to go abroad being considered as part of the broader right to personal liberty.


In [33]:
query2 = "What are Personal Liberty: Procedure Established by Law?"

response = rag_chain2.invoke(query2)
print(response)

 The "Personal Liberty: Procedure Established by Law" refers to a legal procedure that must be fair, just, and reasonable when restricting a person's personal liberty. This means that any law or rule used to deprive someone of their freedom (such as arrest or detention) should not be arbitrary, oppressive, or fanciful, but should follow a proper and well-defined process.


In [36]:
query2 = "What are Personal Liberty: Procedure Established by Law?"

response = rag_chain3.invoke(query2)
print(response)

**Relevant part of the document:** 

According to Article 21, "The procedure prescribed by law has to be fair, just and reasonable, not fanciful, oppressive or arbitrary."

**In simple terms:** This means that when a person is deprived of their liberty (i.e., arrested or detained), there must be a clear and fair process established by law. The process should not be overly restrictive or unfair.

**Obligations/Risks/Important Conditions:**

* There must be a procedure in place for depriving someone of their liberty.
* This procedure must be fair, just, and reasonable.
* It cannot be arbitrary or oppressive.

**What the user is agreeing to:** Not found in this specific part of the document. However, it's implied that governments or authorities are committing to establish and follow a fair process when dealing with personal liberties.


In [40]:
query2 = "What are Personal Liberty: Procedure Established by Law?"

response = rag_chain4.invoke(query2)
print(response)

The document discusses a legal case that determined going abroad is part of personal liberty, which means you have rights to leave your country if the law allows it. The Supreme Court said this right can't be taken away unless there are clear and fair rules in place for doing so. Also mentioned was an international agreement ensuring people deprived of their freedom should always know where they are being held and why, with protections against unfair treatment or punishment while seeking information about these conditions.


In [41]:
q='''This Gift Deed is made and executed on this ____ day of __________, 20____ at ____________________.
BETWEEN:
Mr./Ms. ____________________________, son/daughter/spouse of ____________________________, residing at ________________________________________________, hereinafter referred to as the "DONOR" (which expression shall, unless excluded by or repugnant to the context, be deemed to include his/her heirs, executors, administrators, and assigns) of the ONE PART.
AND:
[Name of Person/Charity/Trust], [if charity, specify registration number], having its address at ________________________________________________, hereinafter referred to as the "DONEE" (which expression shall, unless excluded by or repugnant to the context, be deemed to include its successors and assigns) of the OTHER PART.
WHEREAS:
The Donor is the absolute owner of the self-acquired property/assets described in Schedule A (Immovable) and Schedule B (Movable) attached herewith, which constitutes the entirety of the Donor’s property.
The Donor has no living dependents (or has made separate provisions for dependents) and is disposing of this property out of free will, affection, and philanthropy.
The Donor desires to gift the said property to the Donee, and the Donee has agreed to accept the same.
NOW THIS DEED WITNESSETH AS UNDER:
VOLUNTARY GIFT: That in consideration of natural love and affection (or, if to charity, "altruistic intent"), the Donor hereby freely and voluntarily transfers, conveys, and assigns to the Donee, by way of gift, all of the property described in the attached Schedules.
IRREVOCABLE TRANSFER: This gift is absolute and irrevocable. The Donor hereby relinquishes all rights, titles, interests, and claims over the property.
OWNERSHIP & CLEAR TITLE: The Donor covenants that the property is self-acquired, free from all encumbrances, liens, mortgages, or legal disputes, and the Donor has the full authority to transfer it.
POSSESSION: The Donor has delivered complete, vacant, and physical possession of the property to the Donee on this day.
DONE ACCEPTANCE: The Donee hereby accepts the gift of the property and acknowledges receipt of possession.
EXPENSES & TAXES: All costs related to stamp duty, registration, and transfer of title shall be borne by the [Donor/Donee].
SCHEDULE A (Immovable Property)
(Detailed description: Address, boundary, survey number, area, type of construction)
SCHEDULE B (Movable Property)
(Detailed description: Bank account numbers, vehicle details, jewelry, shares, cash)
IN WITNESS WHEREOF, the Donor and Donee have signed this deed on the day and year first above written.
SIGNATURES:
___________________ (Signature)
DONOR
___________________ (Signature)
DONEE (or Authorized Signatory)
WITNESSES:
(Name and Address): __________________________
(Name and Address): __________________________
 what is this agreement about '''

In [42]:
res=rag_chain.invoke(q)
print(res)

This agreement is a formal transfer of land – essentially, the donor is giving the donee ownership of a property. Here’s a breakdown:

**What it means:**

*   **Donor:** The person giving the land.
*   **Donee:** The person receiving the land.
*   **Gift:** The donor is giving the land to the donee, completely and unconditionally.
*   **Schedule A & B:** These are detailed descriptions of the land, including its location and any existing issues.

**Obligations, Risks, and Important Conditions:**

*   **Obligations:** The donor promises to build a temple on the land, and the donee will use the land for worship.
*   **Risks:** The donor is giving up all rights to the land. The donee will have full ownership.
*   **Important Conditions:** The donor must transfer the land to the donee within one year, and the donee must not use the land for any other purpose.

**What the user is agreeing to:**

The user is agreeing to that the donor will transfer the land to the donee, and the donee will u

In [43]:
res=rag_chain2.invoke(q)
print(res)

 This agreement is a Gift Deed, made on a specific date, between two parties - the Donor and the Donee. The Donor is giving away some self-acquired property to the Donee as a gift. The property includes both immovable (like land or buildings) and movable items (like bank accounts, vehicles, jewelry, shares, cash). Both parties have agreed to this transfer of property, with the Donor promising that the property is free from any encumbrances, liens, mortgages, or legal disputes. The Donor has also handed over physical possession of the property to the Donee. The agreement contains provisions about who will bear the costs related to stamp duty, registration, and transfer of title. Witnesses have signed the deed to confirm its authenticity.


In [44]:
res=rag_chain3.invoke(q)
print(res)

This document is a Gift Deed, which means it's an agreement where one person (the Donor) gives away their property to another person or organization (the Donee) for free.

**Relevant part of the document:**

The relevant parts are:

* The Donor is giving away all their property described in Schedules A and B.
* The gift is made out of love, affection, or philanthropy (if it's a charity).
* The gift is absolute and irrevocable, meaning the Donor can't take it back.
* The Donee accepts the gift and acknowledges receiving possession of the property.

**Obligations, risks, and important conditions:**

* The Donor has no living dependents or has made separate provisions for them.
* The Donor is transferring all their rights, titles, interests, and claims over the property to the Donee.
* The Donee accepts the gift and acknowledges receiving possession of the property.

**What the user is agreeing to:**

The Donor is agreeing to give away their property freely and voluntarily, without expect

In [45]:
res=rag_chain4.invoke(q)
print(res)

This document, called a Deed of Gift, involves one person gifting land to another for the construction of a temple. The donor promises that no other purpose will be used for the property and agrees on certain conditions like completing the temple within a year from when this agreement was made. Both parties have signed it in front of witnesses at a specified place and time, confirming their commitment to these terms.
